# GPT-Neo residual stream — cached to HDF5

Same run as the in-memory example, but with `cache_outputs=True` so
activations stream to an HDF5 file instead of staying in RAM. Downloads on
first run: WikiText-2 (~5 MB) and GPT-Neo 125M weights (~500 MB).

In [5]:
import sys
from pathlib import Path

# Make the repo-local examples._utils package importable when this notebook
# is opened directly from examples/text/, without installing anything extra.
sys.path.insert(0, str(Path.cwd().parents[1]))

from transformers import AutoModelForCausalLM, AutoTokenizer

from examples._utils.text import WikiTextSamples
from nnact import ActivationPipeline
from nnact._model._hooked import HookedModel

MODEL = "EleutherAI/gpt-neo-125m"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token  # GPT-Neo ships without a pad token
model = AutoModelForCausalLM.from_pretrained(MODEL)  # has .logits, real next-token head

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
dataset = WikiTextSamples(tokenizer, n=512, max_length=64)
hooked = HookedModel(model)

# The residual stream: every transformer block, plus the final layer norm.
num_layers = model.config.num_layers
LAYERS = [f"transformer.h.{i}" for i in range(num_layers)] + ["transformer.ln_f"]
print(f"{len(dataset)} passages | {len(LAYERS)} layers: {LAYERS}")

In [ ]:
# cache_outputs=True streams every batch's activations to an HDF5 file at
# cache_dir/activations.h5 instead of accumulating them in memory. Passing a
# tokenizer for a "token" pipeline also attaches token-level metadata:
# token_ids and decoded tokens.
pipeline = ActivationPipeline(
    LAYERS,
    output_type="token",
    tokenizer=tokenizer,
    cache_outputs=True,
    cache_dir=Path.cwd() / "runs" / "04_gpt2_residual_stream_cache",
)
result = pipeline.run(dataset, batch_size=16, model_fn=lambda: model)
activations = result.dataset
print(result.metadata)

[2026-09-18 21:05:40,029][nnact._pipeline][INFO] Starting run: model=GPTNeoForCausalLM, output_type=token


activations[1/32]   3%|3          [00:00<?]

[2026-09-18 21:05:42,941][nnact._pipeline][INFO] Run finished: 512 samples in 2.76s


{'model': 'GPTNeoForCausalLM', 'output_type': 'token', 'layer_names': ['transformer.h.0', 'transformer.h.1', 'transformer.h.2', 'transformer.h.3', 'transformer.h.4', 'transformer.h.5', 'transformer.h.6', 'transformer.h.7', 'transformer.h.8', 'transformer.h.9', 'transformer.h.10', 'transformer.h.11', 'transformer.ln_f'], 'num_samples': 512, 'started_at': '2026-09-18T19:05:40.029430+00:00', 'finished_at': '2026-09-18T19:05:42.941169+00:00', 'duration_seconds': 2.755149196003913, 'env': {'python_version': '3.13.13', 'hostname': 'zani'}}


In [ ]:
# Slicing by offsets pulls out one passage's own tokens across every layer.
import numpy as np

offsets = activations.offsets
start, end = int(offsets[0]), int(offsets[1])
sample_activations = {
    name: activations.activations(name)[start:end] for name in activations.layer_names
}
print(
    "first passage ->",
    len(sample_activations),
    "layers,",
    tuple(next(iter(sample_activations.values())).shape),
    "each,",
)

# Residual stream norm grows with depth - the usual GPT-2/GPT-Neo picture.
for name, tensor in sample_activations.items():
    print(f"  {name:<18} mean L2 norm = {np.linalg.norm(tensor, axis=-1).mean():6.2f}")

first passage -> 13 layers, (64, 768) each,
  transformer.h.0    mean L2 norm = 552.06
  transformer.h.1    mean L2 norm = 1088.06
  transformer.h.10   mean L2 norm = 1348.88
  transformer.h.11   mean L2 norm = 918.30
  transformer.h.2    mean L2 norm = 1409.31
  transformer.h.3    mean L2 norm = 1481.68
  transformer.h.4    mean L2 norm = 1530.43
  transformer.h.5    mean L2 norm = 1575.68
  transformer.h.6    mean L2 norm = 1643.78
  transformer.h.7    mean L2 norm = 1643.67
  transformer.h.8    mean L2 norm = 1688.36
  transformer.h.9    mean L2 norm = 1528.06
  transformer.ln_f   mean L2 norm =  71.19
